# 动手练习：网络上的传播与免疫

来自《生物医学网络科学》（Biomedical Network Science）。 参考答案版。

## 任务 1：用度分布估计流行病阈值

异质平均场给出 SIR 的阈值：每条边的传播性 $T$ 需大于
$$T_c=\frac{\langle k\rangle}{\langle k^2\rangle-\langle k\rangle}.$$
本练习用离散时间 SIR：每一步，每个感染者以概率 $\beta$ 感染每个易感邻居，然后以概率 $\gamma$ 恢复（移除）。一个感染者在整个感染期内把病传给某个邻居的概率（传播性）为
$$T=1-\frac{\gamma(1-\beta)}{1-(1-\beta)(1-\gamma)}.$$

下面已经生成节点数和边数都相同的 BA 网络 `G_ba` 与 ER 网络 `G_er`。请计算：

- `T`：$\beta=0.05$、$\gamma=0.3$ 时的传播性；
- `Tc_er`、`Tc_ba`：两个网络的 $T_c$（用实际度序列的一阶、二阶矩）。

期望看到：两网平均度相同（约 6），但 BA 的 $\langle k^2\rangle$ 大得多，$T_c$ 明显更低；当前的 $T$ 位于两者之间。

> **提示：** `k = np.array([d for _, d in G.degree()], dtype=float)`，`k.mean()` 是 $\langle k\rangle$，`(k**2).mean()` 是 $\langle k^2\rangle$。

In [ ]:
# 准备（直接运行）
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

N, m = 1000, 3
G_ba = nx.barabasi_albert_graph(N, m, seed=1)
G_er = nx.gnm_random_graph(N, G_ba.number_of_edges(), seed=1)
beta, gamma = 0.05, 0.3

In [ ]:
def sir_threshold(G):
    k = np.array([d for _, d in G.degree()], dtype=float)
    return k.mean() / ((k**2).mean() - k.mean())


T = 1 - gamma * (1 - beta) / (1 - (1 - beta) * (1 - gamma))
Tc_er = sir_threshold(G_er)
Tc_ba = sir_threshold(G_ba)
print("传播性 T = %.3f" % T)
for name, G, Tc in [("ER", G_er, Tc_er), ("BA", G_ba, Tc_ba)]:
    k = np.array([d for _, d in G.degree()], dtype=float)
    print("%s: <k>=%.2f, <k^2>=%.1f, T_c=%.3f -> %s" % (name, k.mean(), (k**2).mean(), Tc,
          "T 高于阈值，可能暴发" if T > Tc else "T 低于阈值"))

In [ ]:
# 核对（不用修改）
assert T is not None and Tc_er is not None and Tc_ba is not None, "T、Tc_er、Tc_ba 还没有赋值"
assert abs(T - 0.149) < 0.002, "传播性 T 应约为 0.149；检查公式 T = 1 − γ(1−β)/(1 − (1−β)(1−γ)) 的括号"
_k = np.array([d for _, d in G_er.degree()], dtype=float)
assert abs(Tc_er - _k.mean() / ((_k**2).mean() - _k.mean())) < 1e-6, "Tc_er 不对：SIR 阈值的分母是 <k^2> − <k>，不是 <k^2>（那是 SIS 阈值）"
assert 0.14 < Tc_er < 0.19, "ER 网络的 T_c 应接近 1/<k> ≈ 0.17；检查是否用了度的平方的平均"
assert Tc_ba < Tc_er - 0.03, "BA 网络的 T_c 应明显低于 ER（约 0.07 对 0.16）；检查 Tc_ba 是否用的是 G_ba"
assert Tc_ba < T < Tc_er, "当前参数的 T 应介于两者之间：高于 BA 的阈值、低于 ER 的阈值"
print("✓ 任务 1 通过")

## 任务 2：网络上的离散时间 SIR 模拟

函数 `sir` 已经写好：返回每一步的 S、I、R 比例和最终移除比例。请在 ER 与 BA 两个网络上各跑 30 次（每次随机选 1 个种子），把最终规模分别存为长度 30 的列表 `sizes_er`、`sizes_ba`，并计算 BA 上“大暴发”（最终规模超过 10%）的比例 `p_big_ba`。

期望看到：同样的 $\beta,\gamma$ 下，ER 上几乎从不暴发（任务 1 中 $T<T_c$），BA 上有一部分模拟（约两成）会暴发到人群的两三成。即使在阈值以上，也有不少模拟在早期就熄灭了：最终规模呈“双峰”，所以只报告平均值会掩盖这一点。

> **提示：** `sir(G, beta, gamma, [rng.integers(N)], rng)[1]` 是一次模拟的最终规模。

In [ ]:
# 准备（直接运行）
def sir(G, beta, gamma, seeds, rng, max_steps=500):
    """离散时间 SIR。返回 (每一步的 S, I, R 比例数组, 最终移除比例)。"""
    n = G.number_of_nodes()
    nbrs = [list(G.neighbors(v)) for v in range(n)]
    state = np.zeros(n, dtype=int)          # 0=S, 1=I, 2=R
    state[list(seeds)] = 1
    hist = []
    for _ in range(max_steps):
        infected = np.flatnonzero(state == 1)
        hist.append([(state == 0).mean(), (state == 1).mean(), (state == 2).mean()])
        if infected.size == 0:
            break
        new_inf = set()
        for u in infected:
            for v in nbrs[u]:
                if state[v] == 0 and rng.random() < beta:
                    new_inf.add(v)
        recover = infected[rng.random(infected.size) < gamma]
        state[recover] = 2
        state[list(new_inf)] = 1
    return np.array(hist), (state == 2).mean()


rng = np.random.default_rng(0)

In [ ]:
sizes_er = [sir(G_er, beta, gamma, [rng.integers(N)], rng)[1] for _ in range(30)]
sizes_ba = [sir(G_ba, beta, gamma, [rng.integers(N)], rng)[1] for _ in range(30)]
p_big_ba = float(np.mean(np.array(sizes_ba) > 0.1))
print("ER 平均最终规模 = %.3f，最大 %.3f" % (np.mean(sizes_er), np.max(sizes_er)))
print("BA 平均最终规模 = %.3f，大暴发比例 = %.2f" % (np.mean(sizes_ba), p_big_ba))

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
hist, _ = sir(G_ba, beta, gamma, [0], np.random.default_rng(3))
for j, lab in enumerate("SIR"):
    axes[0].plot(hist[:, j], label=lab)
axes[0].set_xlabel("time step"); axes[0].set_ylabel("fraction"); axes[0].set_title("SIR on BA (seed = node 0)"); axes[0].legend()
axes[1].hist([sizes_er, sizes_ba], bins=np.linspace(0, 1, 21), label=["ER", "BA"])
axes[1].set_xlabel("final size"); axes[1].set_ylabel("runs"); axes[1].legend()
fig.tight_layout()

In [ ]:
# 核对（不用修改）
assert sizes_er is not None and sizes_ba is not None and p_big_ba is not None, "sizes_er、sizes_ba、p_big_ba 还没有赋值"
assert len(sizes_er) == 30 and len(sizes_ba) == 30, "每个网络应各做 30 次模拟"
assert all(0 < s <= 1 for s in list(sizes_er) + list(sizes_ba)), "最终规模是比例，应在 (0, 1] 内；是否取了 sir 返回值的第二项？"
assert np.mean(sizes_er) < 0.05, "ER 上 T 低于阈值，平均最终规模应很小（< 5%）；检查是否把 G_er 和 G_ba 弄反了"
assert np.mean(sizes_ba) > np.mean(sizes_er), "BA 的平均最终规模应大于 ER；检查每次是否随机选了种子"
assert 0 < p_big_ba < 0.9, "BA 上应既有早期熄灭也有大暴发；p_big_ba 应是 sizes_ba 中大于 0.1 的比例（0–1 之间的数），不是个数"
print("✓ 任务 2 通过")

## 任务 3：免疫策略比较

在 BA 网络上接种 10% 的节点（接种者从网络中删除），比较三种策略：随机免疫、目标免疫（度最高者）、熟人免疫（随机选一人，接种他的一个随机邻居，重复直到凑满 `n_vac` 个**不同**的人）。

辅助函数 `evaluate(G, removed)` 返回删除后剩余网络的 $(T_c,\ \text{巨连通分支比例})$。请：

1. 构造熟人免疫的接种名单 `acq_set`（`n_vac` 个互不相同的节点）；
2. 对三种策略分别调用 `evaluate`，得到 `Tc_random, giant_random`、`Tc_target, giant_target`、`Tc_acq, giant_acq`。

期望看到：随机免疫后 $T_c$ 仍低于任务 1 的 $T\approx0.149$；目标免疫把 $T_c$ 提到远高于 $T$；熟人免疫把 $T_c$ 提到 $T$ 附近、明显高于随机免疫，而它完全不需要知道谁是枢纽。

> **提示：** `u = int(rng.integers(N))`，`nb = list(G_ba.neighbors(u))`，`nb[rng.integers(len(nb))]` 加入一个 `set`。

In [ ]:
# 准备（直接运行）
def evaluate(G, removed):
    H = G.copy()
    H.remove_nodes_from(removed)
    k = np.array([d for _, d in H.degree()], dtype=float)
    Tc = k.mean() / ((k**2).mean() - k.mean())
    giant = max(len(c) for c in nx.connected_components(H)) / G.number_of_nodes()
    return Tc, giant


rng = np.random.default_rng(1)
n_vac = int(0.1 * N)
random_set = list(rng.choice(N, n_vac, replace=False))
target_set = [v for v, _ in sorted(G_ba.degree(), key=lambda x: -x[1])[:n_vac]]

In [ ]:
acq = set()
while len(acq) < n_vac:
    u = int(rng.integers(N))
    nb = list(G_ba.neighbors(u))
    acq.add(nb[rng.integers(len(nb))])
acq_set = list(acq)

Tc_random, giant_random = evaluate(G_ba, random_set)
Tc_target, giant_target = evaluate(G_ba, target_set)
Tc_acq, giant_acq = evaluate(G_ba, acq_set)
deg = dict(G_ba.degree())
for name, S, Tc, gc in [("随机", random_set, Tc_random, giant_random),
                        ("目标", target_set, Tc_target, giant_target),
                        ("熟人", acq_set, Tc_acq, giant_acq)]:
    print("%s免疫: 被接种者平均度 %.1f, 剩余网络 T_c = %.3f, 巨分支比例 = %.3f"
          % (name, np.mean([deg[v] for v in S]), Tc, gc))

In [ ]:
# 核对（不用修改）
assert acq_set is not None, "acq_set 还没有赋值"
assert len(set(acq_set)) == n_vac == len(acq_set), "熟人免疫名单应包含 n_vac 个互不相同的节点；用 set 去重，循环到集合大小达到 n_vac"
_deg = dict(G_ba.degree())
assert np.mean([_deg[v] for v in acq_set]) > 1.5 * np.mean([_deg[v] for v in random_set]), \
    "被提名熟人的平均度应明显高于随机抽到的人（朋友悖论）；检查接种的是邻居而不是随机选出的那个人"
assert None not in (Tc_random, Tc_target, Tc_acq, giant_random, giant_target, giant_acq), "三种策略的 T_c 与巨分支比例都要算出来"
assert Tc_random < T, "随机免疫 10% 后 T_c 应仍低于 T ≈ 0.149；检查 Tc_random 是否用的是 random_set"
assert Tc_target > Tc_acq > Tc_random, "T_c 的顺序应为 目标免疫 > 熟人免疫 > 随机免疫；检查各变量是否对应正确的名单"
assert giant_target < giant_random, "目标免疫后的巨分支应比随机免疫后小"
print("✓ 任务 3 通过")

## 任务 4：扩散中心性与独立级联

扩散中心性 $DC=\left[\sum_{t=1}^{T}(qA)^t\right]\mathbf 1$。在空手道俱乐部网络上取 $q=0.15$、$T=3$：

1. 计算长度为 34 的扩散中心性向量 `DC`；
2. `spread` 已经给出：用独立级联模型（每条边一次机会、概率 $q$）模拟每个节点作为单种子的平均传播规模（各 200 次）。计算度与 `spread`、DC 与 `spread` 的 Spearman 相关系数 `rho_deg`、`rho_dc`；
3. 列出 DC 排名前 5 的节点 `top5_dc`（列表，从大到小）。

期望看到：两者与模拟结果都高度相关，DC 更高一些，因为它数的是 $T$ 步内的传播路径，而不只是直接邻居。

> **提示：** `M = q * A`，用循环累加 `M`、`M @ M`、`M @ M @ M`，再乘以 `np.ones(n)`；`spearmanr(x, y)[0]` 是相关系数。

In [ ]:
# 准备（直接运行）
from scipy.stats import spearmanr

K = nx.karate_club_graph()
A = nx.to_numpy_array(K, weight=None)
n = A.shape[0]
q, Tmax = 0.15, 3
deg_k = A.sum(1)


def ic_spread(G, seed, q, rng):
    active, frontier = {seed}, [seed]
    while frontier:
        nxt = []
        for u in frontier:
            for v in G.neighbors(u):
                if v not in active and rng.random() < q:
                    active.add(v)
                    nxt.append(v)
        frontier = nxt
    return len(active)


_rng = np.random.default_rng(0)
spread = np.array([np.mean([ic_spread(K, s, q, _rng) for _ in range(200)]) for s in range(n)])

In [ ]:
M = q * A
P = np.eye(n)
S = np.zeros((n, n))
for _ in range(Tmax):
    P = P @ M
    S += P
DC = S @ np.ones(n)
rho_deg = spearmanr(deg_k, spread)[0]
rho_dc = spearmanr(DC, spread)[0]
top5_dc = np.argsort(-DC)[:5].tolist()
print("度 前5:", np.argsort(-deg_k)[:5].tolist())
print("DC 前5:", top5_dc)
print("IC 规模 前5:", np.argsort(-spread)[:5].tolist())
print("Spearman(度, IC) = %.3f, Spearman(DC, IC) = %.3f" % (rho_deg, rho_dc))

In [ ]:
# 核对（不用修改）
assert DC is not None and rho_deg is not None and rho_dc is not None and top5_dc is not None, "DC、rho_deg、rho_dc、top5_dc 还没有赋值"
_M = q * A
_ref = (_M + _M @ _M + _M @ _M @ _M) @ np.ones(n)
assert np.shape(DC) == (n,), "DC 应是长度为 34 的向量（矩阵乘以全 1 向量）"
assert np.allclose(DC, _ref, rtol=1e-6), "DC 数值不对：应累加 t = 1, 2, 3 三项 (qA)^t，不含 t = 0 的单位阵，且每一项都乘 q^t"
assert list(top5_dc)[:2] == [33, 0], "DC 最大的两个节点应是 33 和 0（两派的首领）；是否从大到小排序？"
assert 0.7 < rho_deg < 1 and 0.7 < rho_dc < 1, "两个 Spearman 相关都应在 0.7–1 之间；检查是否与 spread 比较"
assert rho_dc >= rho_deg - 0.02, "扩散中心性与模拟结果的相关应不低于度"
print("✓ 任务 4 通过")

## 任务 5：开放分析：医院病区接触网络上的感染控制

`data/hospital_lyon_contacts.tsv` 是法国里昂一家大学医院老年科病区 4 天的可穿戴传感器接触记录（Vanhems 等 2013，PLoS ONE；SocioPatterns，CC BY-NC-SA 3.0）。每行 `t i j Si Sj`：$i$ 与 $j$ 在区间 $[t-20\,\mathrm{s},t]$ 内处于接触状态；身份 NUR 为护士与护理员，PAT 为患者，MED 为医生，ADM 为行政人员。

setup 已把记录累积成加权静态网络 `H`（边权 `weight` 为累积接触秒数），并给出 `role`（节点 → 身份）。请自己设计一个小分析，回答：**如果只能对 5 个人加强防护（相当于从网络中删去），选谁最能减少院内传播？** 至少比较 3 种策略，例如不干预、随机 5 人、度最高 5 人、累积接触时长最高 5 人、只选患者或只选医护等。

建议做法：把边权换成传播性 $T_{ij}=1-e^{-\beta w_{ij}}$（$\beta$ 自己选，比如每小时接触传播概率约 0.5 对应 $\beta=\ln 2/3600$），用键渗流模拟 SIR：每次按 $T_{ij}$ 独立保留每条边，随机选一个未被删去的种子，它所在连通分支的大小就是最终感染人数。每种策略重复几百次取平均。

结果存为字典 `final_size`：键是策略名（字符串），值是平均最终感染比例（相对于 75 人）。另把你选的 $\beta$ 存为 `beta_hosp`。最后用两三句话写下结论与局限（例如：静态累积网络忽略了接触的时间顺序，会高估传播）。

In [ ]:
# 准备（直接运行）
import pandas as pd

raw = pd.read_csv("data/hospital_lyon_contacts.tsv", sep="\t", header=None, names=["t", "i", "j", "Si", "Sj"])
role = dict(zip(raw.i, raw.Si)) | dict(zip(raw.j, raw.Sj))
pairs = pd.DataFrame({"a": raw[["i", "j"]].min(axis=1), "b": raw[["i", "j"]].max(axis=1)})
w = pairs.groupby(["a", "b"]).size() * 20          # 每对人的累积接触秒数
H = nx.Graph()
H.add_nodes_from(role)
H.add_weighted_edges_from((int(a), int(b), float(s)) for (a, b), s in w.items())
print("节点 %d，边 %d；身份构成 %s" % (H.number_of_nodes(), H.number_of_edges(),
      pd.Series(role).value_counts().to_dict()))

In [ ]:
beta_hosp = np.log(2) / 3600
final_size = {}
rng_h = np.random.default_rng(2026)
nodes = list(H.nodes())
N_h = len(nodes)


def outbreak(H, removed, beta, runs, rng):
    keep = [v for v in H.nodes() if v not in set(removed)]
    edges = [(u, v, d["weight"]) for u, v, d in H.edges(data=True) if u in keep and v in keep]
    Tij = np.array([1 - np.exp(-beta * wt) for _, _, wt in edges])
    sizes = []
    for _ in range(runs):
        F = nx.Graph()
        F.add_nodes_from(keep)
        F.add_edges_from(e[:2] for e, x in zip(edges, rng.random(len(edges)) < Tij) if x)
        seed = keep[rng.integers(len(keep))]
        sizes.append(len(nx.node_connected_component(F, seed)) / N_h)
    return float(np.mean(sizes))


strength = dict(H.degree(weight="weight"))
degree = dict(H.degree())
strategies = {
    "不干预": [],
    "随机5人": None,
    "度最高5人": sorted(nodes, key=lambda v: -degree[v])[:5],
    "接触时长最高5人": sorted(nodes, key=lambda v: -strength[v])[:5],
    "接触时长最高5名患者": sorted([v for v in nodes if role[v] == "PAT"], key=lambda v: -strength[v])[:5],
}
for name, S in strategies.items():
    if S is None:   # 随机策略：每次重新抽 5 人，共 20 组，每组 20 次
        final_size[name] = float(np.mean([outbreak(H, list(rng_h.choice(nodes, 5, replace=False)), beta_hosp, 20, rng_h)
                                          for _ in range(20)]))
    else:
        final_size[name] = outbreak(H, S, beta_hosp, 400, rng_h)
    who = "" if not S else "（" + "、".join(role[v] for v in S) + "）"
    print("%-12s 平均最终感染比例 %.3f %s" % (name, final_size[name], who))

fig, ax = plt.subplots(figsize=(6, 3))
ax.barh(list(final_size), list(final_size.values()))
ax.set_xlabel("mean final size (fraction of 75)")
fig.tight_layout()
# 结论示例：按累积接触时长或度选出的 5 人几乎都是医护人员，删去他们使平均最终规模下降最多；
# 随机 5 人和只选患者的效果有限。局限：静态累积网络忽略了时间顺序（会高估传播）、β 是假设值、只有 4 天数据。

In [ ]:
# 核对（不用修改）
assert beta_hosp is not None and beta_hosp > 0, "beta_hosp 应是一个正的传播速率（每秒）"
assert isinstance(final_size, dict) and len(final_size) >= 3, "final_size 应是至少包含 3 种策略的字典"
assert all(isinstance(k_, str) for k_ in final_size), "final_size 的键应是策略名（字符串）"
assert all(0 <= float(v) <= 1 for v in final_size.values()), "final_size 的值是相对于 75 人的比例，应在 0–1 之间"
print("✓ 任务 5 通过")